# PDF loading

In [1]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader

documents = []
data_dir = Path("./data")

for file in data_dir.iterdir():

    if file.suffix.lower() == ".txt":
        print(f"Loading TXT: {file}")
        documents.extend(TextLoader(str(file)).load())

    elif file.suffix.lower() == ".pdf":
        print(f"Loading PDF: {file}")
        documents.extend(PyPDFLoader(str(file)).load())

print(f"Total documents loaded: {len(documents)}")


/tmp/ipykernel_2127/638227778.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, PyPDFLoader
/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading PDF: data/Retrieval-Augmented_Generation_RAG.pdf
Total documents loaded: 12


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks=splitter.split_documents(documents)
print("Splitter created Successfully")
print(f"length of chunks:{len(chunks)}")


Splitter created Successfully
length of chunks:152


In [ ]:
from sentence_transformers import SentenceTransformer

model= SentenceTransformer("all-MiniLM-L6-v2")
texts=[chunk.page_content for chunk in chunks]

embeddings=model.encode(
            texts,
            convert_to_numpy=True
        )
print(embeddings.shape)
 
 #---storing in vector db--
import chromadb
client = chromadb.PersistentClient(path='./chroma_db')

collection= client.get_or_create_collection( name= "documents")

collection.add(
     ids=[f"doc_{i}" for i in range(len(texts))],
    documents=texts,
    embeddings=embeddings.tolist()
    # ids=[f"doc_{i}" for i in range(len(texts))]
    # metadatas=[chunk.metadata for chunk in chunks]
)

print("Documents loaded successfully!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 312.96it/s]


(152, 384)
Documents loaded successfully!


# dense-search

In [5]:
def dense_search(query, top_k=10):
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    ).tolist()

    results =collection.query(
        query_embeddings=[query_embedding],
        n_results= top_k
    )
    return results

In [6]:
results=dense_search( "What is Retrieval Agumented Generation and how does it work?")

print(results["documents"][0][0][:500])

sions. Finally, we discuss important research avenues for
the BISE community by highlighting implications that
emerge as a consequence of using RAG architectures.
2 Retrieval-Augmented Generation (RAG)
2.1 Fundamental Framework
T h ec o r ei d e ao fR A Gi st oc o m bine the generative capabil-
ities of LLMs with external knowledge retrieved from a
separate database (e.g., an organizational database) (Lewis
et al. 2020). While Lewis et al. (2020) acknowledge previous


# BM25 (sparse search)

In [11]:
from rank_bm25 import BM25Okapi
token_corpus=[
    doc.split()
    for doc in texts
 ]

bm25=BM25Okapi(
    token_corpus
)

def bm25_search(query,top_k=10):
    token_query = query.split()
    scores = bm25.get_scores(token_query)
    ranked = sorted( enumerate(scores), key=lambda x: x[1], reverse= True)

    results= []
    for idx, score in ranked[:top_k]:
        results.append(
            { "document":texts[idx],
             "score":score
             }
        )
    return results

In [12]:
bm25_results= bm25_search(" What is Retrieval Agumented Generation and how does it work?")
print(bm25_results[0]["document"][:500])

issue.
Retrieval effectiveness The effectiveness of a RAG architecture depends on how effectively the retrieval mechanism works.
This includes the effectiveness of the document ranking (i.e., are the most relevant documents ranked
ﬁrst?) and how well the retrieval process performs.
Fig. 3 Research questions related to RAG
123
558 M. Klesel, H. F. Wittmann: Retrieval-Augmented Generation (RAG), Bus Inf Syst Eng 67(4):551–561 (2025)


# hybrid search

In [ ]:
def hybrid_search(query,top_k=20):
    dense_results = dense_search(query,top_k=10)
    bm25_results = bm25_search(query, top_k=10)
    
    docs = [ ]
    for doc in dense_results["documents"][0]:
        docs.append(doc)
        
    for item in bm25_results:
        docs.append(item["document"])
    docs= list(dict.fromkeys(docs))
    return docs[:top_k]

# before re-ranking

In [36]:
query ="What is Retrieval Agumented Generation and how does it work?"
docs = hybrid_search(query)

print("Retrieved Documents:", len(docs))

for i, doc in enumerate(docs[:10]):
    print(f"\n===== Retrieved Chunk {i+1} =====")
    print(doc[:300])

Retrieved Documents: 19

===== Retrieved Chunk 1 =====
sions. Finally, we discuss important research avenues for
the BISE community by highlighting implications that
emerge as a consequence of using RAG architectures.
2 Retrieval-Augmented Generation (RAG)
2.1 Fundamental Framework
T h ec o r ei d e ao fR A Gi st oc o m bine the generative capabil-
itie

===== Retrieved Chunk 2 =====
et al. 2020). While Lewis et al. (2020) acknowledge previous
work on the integration of external data (Guu et al. 2020;
Karpukhin et al. 2020; Perez et al. 2019), they coined the
term ‘ ‘Retrieval-Augmented Generation (RAG)’ ’ and pro-
posed a general framework that leverages the strength of pre-
tr

===== Retrieved Chunk 3 =====
Khan AA, Hasan MT, Kemell KK, Rasku J, Abrahamsson P (2024)
Developing retrieval augmented generation (RAG) based LLM
systems from PDFs: an experience report. https://doi.org/10.
48550/ARXIV.2410.15944
Lewis P, Perez E, Piktus A, Petroni F, Karpukhin V, Goyal N, Ku ¨ttler
H, Lewis

In [ ]:
docs=hybrid_search("What is Retrieval Agumented Generation and how does it work?")
print(len(docs))

19


# What is Cohere Rerank?

Cohere Rerank is a model that takes:
A user query,
A list of retrieved documents/chunks
and then re-orders the chunks from most relevant to least relevant.

In [ ]:
import cohere

coh=cohere.Client("API_key")


def rerank_documents(query,docs):
    response=coh.rerank(
        model="rerank-v3.5",
        query=query,
        documents=docs,
        top_n=5
    )
    top_docs=[]
    for result in response.results:
       top_docs.append(
         docs[result.index]
    )
    return top_docs

In [37]:
query = "What is Retrieval Agumented Generation and how does it work?"

docs = hybrid_search(query)

print("Retrieved:", len(docs))

top_docs = rerank_documents(query, docs)

print("Reranked:", len(top_docs))


Retrieved: 19
Reranked: 5


#  after re-ranking

In [ ]:

for i, doc in enumerate(top_docs):
    print(f"\n===== RERANKED CHUNK {i+1} =====")
    print(doc[:500])


===== RERANKED CHUNK 1 =====
real-world facts or user inputs ’ ’ (Ji et al. 2023, p. 1).
1
Hallucinations are particularly critical, because they
undermine the trustworthiness of the results and have been
observed in various scenarios, such as multilingual use of
LLMs (Guerreiro et al. 2023), or in context-speciﬁc situa-
tions, such as medicine (Pal et al. 2023).
Retrieval-augmented generation (RAG) has been pro-
posed as a new framework for AI that seeks to integrate
additional knowledge, such as organizational data, and

===== RERANKED CHUNK 2 =====
et al. 2020). While Lewis et al. (2020) acknowledge previous
work on the integration of external data (Guu et al. 2020;
Karpukhin et al. 2020; Perez et al. 2019), they coined the
term ‘ ‘Retrieval-Augmented Generation (RAG)’ ’ and pro-
posed a general framework that leverages the strength of pre-
trained parametric memory (i.e., the LLM) with non-para-
metric memory (i.e., a separate database) as a new way to
improve the performance for 

In [24]:
top_docs= rerank_documents("What is Retrieval Agumented Generation and how does it work?", docs)
print(top_docs)

['real-world facts or user inputs ’ ’ (Ji et al. 2023, p. 1).\n1\nHallucinations are particularly critical, because they\nundermine the trustworthiness of the results and have been\nobserved in various scenarios, such as multilingual use of\nLLMs (Guerreiro et al. 2023), or in context-speciﬁc situa-\ntions, such as medicine (Pal et al. 2023).\nRetrieval-augmented generation (RAG) has been pro-\nposed as a new framework for AI that seeks to integrate\nadditional knowledge, such as organizational data, and', 'et al. 2020). While Lewis et al. (2020) acknowledge previous\nwork on the integration of external data (Guu et al. 2020;\nKarpukhin et al. 2020; Perez et al. 2019), they coined the\nterm ‘ ‘Retrieval-Augmented Generation (RAG)’ ’ and pro-\nposed a general framework that leverages the strength of pre-\ntrained parametric memory (i.e., the LLM) with non-para-\nmetric memory (i.e., a separate database) as a new way to\nimprove the performance for knowledge-intensive tasks.', 'CATCHWORD

In [27]:
def build_context(top_docs):
    return  "\n\n".join(top_docs)
context = build_context(top_docs)

In [28]:
import os
import boto3 
from dotenv import load_dotenv

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
load_dotenv("myenv.env")

bedrock_runtime = boto3.client(
     service_name="bedrock-runtime",
     region_name=AWS_REGION,
     aws_access_key_id=AWS_ACCESS_KEY_ID,
     aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)
MODEL_ID = "amazon.nova-micro-v1:0"

In [39]:
import json

def generate_answer(query, context):

    prompt = f"""
You are a helpful  assistant.

Answer the  question using provided context.

Context:
{context}

Question:
{query}
"""

    request_body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ]
            }
        ]
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(request_body)
    )

    response_body = json.loads(
        response["body"].read()
    )

    answer=response_body["output"]["message"]["content"][0]["text"]
    print(answer)

In [32]:
generate_answer(
    query="What is Retrieval Agumented Generation and how does it work?",
    context=context
)

Retrieval-Augmented Generation (RAG) is an advanced framework in artificial intelligence that aims to enhance the performance of knowledge-intensive tasks by combining the generative capabilities of large language models (LLMs) with external knowledge retrieved from a separate database, such as an organizational database. 

Here's how it works:

1. **Fundamental Framework**: 
   - **Generative Capabilities of LLMs**: RAG leverages the powerful text generation abilities of LLMs, which have been pre-trained on vast amounts of data and can generate coherent and contextually relevant text.
   - **External Knowledge**: It augments these generative capabilities by integrating information from an external knowledge source, which can be organizational data, a database, or any other structured data source.

2. **Integration Process**:
   - **Input Query**: When a user inputs a query, the RAG system first retrieves relevant documents or information from the external knowledge source based on the

In [7]:
import numpy as np
from rank_bm25 import BM25Okapi

# Query to test the retrievers
query = "What is retrieval augmented generation and how does it work?"

# Use the chunks already created in the notebook
corpus = [doc.page_content for doc in documents]

if not corpus:
    raise ValueError("No documents available for BM25/dense search. Run the chunking cells first.")

# -------------------------
# 1) BM25 search
# -------------------------
tokenized_corpus = [text.lower().split() for text in corpus]
bm25 = BM25Okapi(tokenized_corpus)

query_tokens = query.lower().split()
bm25_scores = bm25.get_scores(query_tokens)

bm25_top_idx = np.argsort(bm25_scores)[::-1][:5]
bm25_results = [
    {"index": int(i), "score": float(bm25_scores[i]), "text": corpus[i]}
    for i in bm25_top_idx
]

print("BM25 top matches:")
for rank, item in enumerate(bm25_results, start=1):
    print(f"\n[{rank}] score={item['score']:.4f}, index={item['index']}")
    print(item["text"][:400].replace("\n", " "))

# -------------------------
# 2) Dense (embedding) search
# -------------------------
# Use the existing sentence-transformer model from cell 1
dense_vectors = embedding_model.encode(
    corpus,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_vector = embedding_model.encode(
    query,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Cosine similarity (because vectors are normalized)
dense_scores = dense_vectors @ query_vector

dense_top_idx = np.argsort(dense_scores)[::-1][:5]
dense_results = [
    {"index": int(i), "score": float(dense_scores[i]), "text": corpus[i]}
    for i in dense_top_idx
]

print("\n\nDense (embedding) top matches:")
for rank, item in enumerate(dense_results, start=1):
    print(f"\n[{rank}] score={item['score']:.4f}, index={item['index']}")
    print(item["text"][:400].replace("\n", " "))

BM25 top matches:

[1] score=10.8771, index=80
the total cost of ownership (Balaguer et al. 2024) 123 M. Klesel, H. F. Wittmann: Retrieval-Augmented Generation (RAG), Bus Inf Syst Eng 67(4):551–561 (2025) 557 Content courtesy of Springer Nature, terms of use apply. Rights reserved. tuning that leads to superior organizational performance? , or To what extent does a RAG-based architecture con- tribute to better IT business alignment? Secondly, 

[2] score=8.4577, index=88
extensive data. New approaches such as RAPTOR (Sarthi et al. 2024) are required to reduce this issue. Retrieval effectiveness The effectiveness of a RAG architecture depends on how effectively the retrieval mechanism works. This includes the effectiveness of the document ranking (i.e., are the most relevant documents ranked ﬁrst?) and how well the retrieval process performs. Fig. 3 Research questi

[3] score=8.1843, index=90
research are Can grounding-based explanations outper- form traditional XAI approaches in improv